#### 앙상블(Ensemble)
- 여러개의 분류모델을 조합해서 더 나은 성능을 내는 방법
- Decision Tree 모델을 증가시켜 나온 랜덤포레스트가 대표적이다.


#### 랜덤포레스트(Random Forest)
##### 랜덤포레스트 훈련방법
- 부트스트랩 샘플 : 중복을 허용하는 샘플링 방법
- 샘플링 후에 샘플을 복구하고 다시 샘플링 하는 방법이다.
- 이와 같이 진행하는 이유는 결정트리에서 과대적랍을 방지할 수 있기 때문이다.
- 각 결정트리에서 나오는 확률의 합을 트리갯수로 나누어서 결정짓는 모델
- 부트스트랩의 샘플링 갯수 : 특성의 갯수의 제곱근

In [1]:
import pandas as pd

In [2]:
wine = pd.read_csv("../Data/wine.csv")
wine.head()

,alcohol,sugar,pH,class
0,9.4,1.9,3.51,0.0
1,9.8,2.6,3.20,0.0
2,9.8,2.3,3.26,0.0
3,9.8,1.9,3.16,0.0
4,9.4,1.9,3.51,0.0


In [3]:
wine.info()

<class 'pandas.DataFrame'>
RangeIndex: 6497 entries, 0 to 6496
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   alcohol  6497 non-null   float64
 1   sugar    6497 non-null   float64
 2   pH       6497 non-null   float64
 3   class    6497 non-null   float64
dtypes: float64(4)
memory usage: 203.2 KB


In [4]:
# Feature와 Target
data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()


In [5]:
# Train과 Test

from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target = \
   train_test_split(
      data,
      target,
      test_size=0.2,
      random_state=42,
      stratify=target
)

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate

In [7]:
rf = RandomForestClassifier(
   n_jobs= -1 # 내 PC의 모든 가용한 자원 사용
)

scores = cross_validate(
   rf,
   train_input,
   train_target,
   return_train_score=True,
   n_jobs=1

)

scores

{'fit_time': array([1.39184999, 1.31971908, 0.86068296, 0.49393797, 0.49575877]),
 'score_time': array([0.08321214, 0.08033395, 0.0676322 , 0.05065823, 0.05174327]),
 'test_score': array([0.90192308, 0.90192308, 0.88065448, 0.89220404, 0.87872955]),
 'train_score': array([0.99807554, 0.99807554, 0.9983165 , 0.998076  , 0.9983165 ])}

In [8]:
print(" Train : ", scores['train_score'].mean())
print(" Valid : ", scores['test_score'].mean())

 Train :  0.9981720130385032
 Valid :  0.8910868438587398


In [9]:
rf.fit(train_input,train_target)
rf.score(test_input,test_target)

0.89

In [10]:
# 중요 Feature
print(rf.feature_importances_)

[0.23846223 0.4963839  0.26515387]


----
#### Extra Tree
- 기본적으로 100갸의 트리를 사용
- 노드 분할시 특성의 제곱근의 갯수를 사용
- 특성의 선택을 랜덤하게 선택한다.
- 특성의 선택을 랜덤하게 하므로 속도는 램덤포레스트보다 빠르다.

In [11]:
from sklearn.ensemble import ExtraTreesClassifier

et = ExtraTreesClassifier(
   n_jobs=-1  # 내 PC의 모든 가용한 자원 사용
)

scores = cross_validate(
   et,
   train_input,
   train_target,
   return_train_score=True,
   n_jobs=1
)

scores

{'fit_time': array([0.71398711, 0.50041294, 0.43069005, 0.66914105, 0.88609481]),
 'score_time': array([0.07838297, 0.07664514, 0.05454183, 0.11265683, 0.11729908]),
 'test_score': array([0.89230769, 0.90576923, 0.88161694, 0.89316651, 0.87680462]),
 'train_score': array([0.99807554, 0.99807554, 0.9983165 , 0.998076  , 0.9983165 ])}

In [12]:
print("Train :", scores['train_score'].mean())
print("Vaild :", scores['test_score'].mean())

Train : 0.9981720130385032
Vaild : 0.8899329977048938


----
#### Gradient Boosting
- 가장 유명한 알고리즘 중 하나이다.
- 경사하강법 처럼 손실함수를 사용.
- 손실함수를 보고 트리를 추가하여 최적의 값을 도출하는 방법이다.
- max depth는 3으로 제어됨 --> 과대적합 방지
- 단점은 손실함수를 확인하고 ㅌ리는 추가하는 모델이므로 n_jobs(병렬처리)를 할 수 없다.

In [13]:
from sklearn.ensemble import GradientBoostingClassifier

gd = GradientBoostingClassifier()

scores = cross_validate(
   gd,
   train_input,
   train_target,
   return_train_score=True,
   n_jobs=-1
)
print("Train :", scores['train_score'].mean())
print("Vaild :", scores['test_score'].mean())

Train : 0.886136193834053
Vaild : 0.8714622047827053


----
#### Histogram Gradient Boosting

- 훈련데이터를 256개의 구간으로 나누어서 훈련시키는 방법
- 특성의 범위가 제한되어 있어 빠른 속도를 제공한다,
- 제한된 구간이므로 과대적합을 방지한다

In [14]:
# from sklearn.experimental import enable_hist_gradient_boosting
# 위의 enable_hist_gradient_boosting이 최종적으로 등록되서 아래의 것으로 등록됨. 고로 아래의 것을 사용하면 됨
from sklearn.ensemble import HistGradientBoostingClassifier

hgb  = HistGradientBoostingClassifier()
scores = cross_validate(
   hgb,
   train_input,
   train_target,
   return_train_score=True,
   n_jobs=-1
)
print("Train :", scores['train_score'].mean())
print("Vaild :", scores['test_score'].mean())

Train : 0.9315949163675888
Vaild : 0.8754999259643148


----
#### LightGBM

- Gradient Boosting에서 출발

In [15]:
from lightgbm import LGBMClassifier


In [16]:
lgb  = LGBMClassifier()
scores = cross_validate(
   hgb,
   train_input,
   train_target,
   return_train_score=True,
   n_jobs=-1
)
print("Train :", scores['train_score'].mean())
print("Vaild :", scores['test_score'].mean())

Train : 0.9315949163675888
Vaild : 0.8754999259643148
